# Agriculture & Climate SLM for Nigerian Smallholder Farmers

Fine-tunes GPT-2 into a small language model that answers farmer-extension questions
(pests, soil health, fertiliser, climate adaptation, livestock, post-harvest, water
management), grounded in the 24 extension factsheets (`documents.csv`) provided with the
challenge.

Pipeline:
1. **Load the data** (`train_qa.csv`, `test_questions.csv`, `documents.csv`)
2. **Process the data**
3. **Tokenize the data with BPE**
4. **Embed the data**
5. **Load the GPT-2 transformer**
6. **Train the model** (each question is trained together with the factsheet it was
   drawn from, via `document_id`, so the model learns to ground its answers)
7. **Evaluate the model** (mean Levenshtein distance -- the challenge's actual scoring
   metric -- plus semantic similarity as a secondary signal)
8. **Make predictions** (topic-filtered document retrieval for the test questions, which
   have no `document_id`, then generation; writes `submission.csv` in the exact
   `QuestionId,Answer` format of `sample_submission.csv`)


In [2]:
from google.colab import files
uploaded = files.upload()  # Upload agriculture-climate-slm-challenge.zip


Saving agriculture-climate-slm-challenge.zip to agriculture-climate-slm-challenge.zip


In [3]:
import os
import re
import glob
import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 1. Load the data

In [4]:
import zipfile

candidate_zips = glob.glob('/content/*.zip')
if not candidate_zips:
    raise FileNotFoundError(
        "No .zip file found in /content. Make sure you ran the upload cell "
        "above and selected agriculture-climate-slm-challenge.zip."
    )
zip_file_path = candidate_zips[0]
output_dir = '/content/agriculture_data'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print(f"'{zip_file_path}' unzipped to '{output_dir}'")
for root, dirs, filenames in os.walk(output_dir):
    for name in filenames:
        print(os.path.join(root, name))


'/content/agriculture-climate-slm-challenge.zip' unzipped to '/content/agriculture_data'
/content/agriculture_data/dataset-metadata.json
/content/agriculture_data/documents.csv
/content/agriculture_data/train_qa.csv
/content/agriculture_data/sample_submission.csv
/content/agriculture_data/test_questions.csv
/content/agriculture_data/baseline_submission.csv


In [5]:
def find_file(filename, root):
    matches = glob.glob(os.path.join(root, '**', filename), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find '{filename}' under '{root}'.")
    return matches[0]

train_df = pd.read_csv(find_file('train_qa.csv', output_dir))
test_df = pd.read_csv(find_file('test_questions.csv', output_dir))
documents_df = pd.read_csv(find_file('documents.csv', output_dir))

print(f"train_qa: {train_df.shape}, test_questions: {test_df.shape}, documents: {documents_df.shape}")
display(train_df.head())
display(test_df.head())
display(documents_df[['document_id', 'title', 'topic', 'crop', 'agro_zone']].head())

ID_COL = 'QuestionId'
ANSWER_COL = 'Answer'  # matches sample_submission.csv exactly
assert ID_COL in test_df.columns, f"Expected an '{ID_COL}' column in test_questions.csv"


train_qa: (45, 7), test_questions: (12, 5), documents: (24, 9)


,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5


,QuestionId,question,topic,crop,agro_zone
0,1001,How should I apply nitrogen to leaching-prone ...,crop_diseases,maize,sub_humid
1,1002,Grass gone in August — feed strategy?,livestock,livestock,semi_arid
2,1003,Red spots under my bean leaves during the rains.,crop_diseases,beans,highland
3,1004,Fresh cow dung on vegetable beds — safe?,fertiliser,general,sub_humid
4,1005,How much compost per hectare?,fertiliser,general,sub_humid


,document_id,title,topic,crop,agro_zone
0,doc_dis_001,Maize nitrogen deficiency symptoms,crop_diseases,maize,sub_humid
1,doc_dis_002,Common bean rust management,crop_diseases,beans,highland
2,doc_dis_003,Cassava mosaic disease,crop_diseases,cassava,sub_humid
3,doc_pes_001,Fall armyworm in maize,pests,maize,semi_arid
4,doc_pes_002,Stem borer in sorghum,pests,sorghum,semi_arid


## 2. Process the data

In [6]:
def clean_text(text):
    if isinstance(text, str):
        text = text.strip()
        text = re.sub(r'\s+', ' ', text)
    return text

def process_qa(df):
    df_clean = df.copy()
    text_cols = [c for c in ('question', 'reference_answer') if c in df_clean.columns]
    df_clean = df_clean.dropna(subset=text_cols)
    for col in text_cols:
        df_clean[col] = df_clean[col].apply(clean_text)
    return df_clean.reset_index(drop=True)

train_df_cleaned = process_qa(train_df)
test_df_cleaned = process_qa(test_df)
documents_df['text'] = documents_df['text'].apply(clean_text)

print(f"Train: {len(train_df)} -> {len(train_df_cleaned)} rows after cleaning")
print(f"Test:  {len(test_df)} -> {len(test_df_cleaned)} rows after cleaning")

# Index documents by id for fast lookup (used when building grounded prompts below)
doc_by_id = documents_df.set_index('document_id')


Train: 45 -> 45 rows after cleaning
Test:  12 -> 12 rows after cleaning


## 3. Tokenize the data with BPE

We train a BPE tokenizer on the full corpus (questions, reference answers, and the
factsheet texts) to inspect vocabulary/length statistics, and use the length statistics
to size `MAX_LENGTH` for GPT-2 fine-tuning below -- factsheets are much longer than the
questions/answers, so this step matters more here than it would for QA text alone.


In [7]:
!pip install -q transformers tokenizers


In [8]:
from tokenizers import ByteLevelBPETokenizer

corpus = (
    train_df_cleaned['question'].tolist()
    + train_df_cleaned['reference_answer'].tolist()
    + test_df_cleaned['question'].tolist()
    + documents_df['text'].tolist()
)
corpus = [str(t) for t in corpus if isinstance(t, str) and t.strip()]

bpe_tokenizer = ByteLevelBPETokenizer()
bpe_tokenizer.train_from_iterator(
    corpus, vocab_size=1500, min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"],
)
# Note: this tiny corpus (45 QA pairs + 24 short factsheets) naturally caps out around
# ~1,300 merges regardless of the vocab_size requested -- that's expected for a dataset
# this size, not a bug.

doc_lengths = [len(bpe_tokenizer.encode(t).ids) for t in documents_df['text']]
qa_lengths = [len(bpe_tokenizer.encode(t).ids)
              for t in train_df_cleaned['question'].tolist() + train_df_cleaned['reference_answer'].tolist()]

print(f"BPE vocabulary size: {bpe_tokenizer.get_vocab_size()}")
print(f"Document token length -- mean: {np.mean(doc_lengths):.1f}, max: {max(doc_lengths)}")
print(f"Q/A token length -- mean: {np.mean(qa_lengths):.1f}, max: {max(qa_lengths)}")

sample = "how should i apply nitrogen to leachingprone maize fields"
print("\nSample text:", sample)
print("BPE tokens:", bpe_tokenizer.encode(sample).tokens)

# Prompt = factsheet text + question + answer, so size MAX_LENGTH off the combined length
MAX_LENGTH = min(320, max(doc_lengths) + max(qa_lengths) + 20)
print(f"\nMAX_LENGTH selected for GPT-2 fine-tuning: {MAX_LENGTH}")


BPE vocabulary size: 1297
Document token length -- mean: 64.5, max: 81
Q/A token length -- mean: 13.0, max: 21

Sample text: how should i apply nitrogen to leachingprone maize fields
BPE tokens: ['h', 'ow', 'Ġshould', 'Ġ', 'i', 'Ġapply', 'Ġnitrogen', 'Ġto', 'Ġleaching', 'prone', 'Ġmaize', 'Ġfields']

MAX_LENGTH selected for GPT-2 fine-tuning: 122


## 4. Embed the data

Sentence embeddings are used for the **test-time document retrieval** in step 8: test
questions have no `document_id`, so we need a way to find the right factsheet to ground
generation in.


In [9]:
from transformers import AutoTokenizer, AutoModel

embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name)
embed_model.eval()

def get_text_embeddings(texts, batch_size=32):
    """Mean-pooled, L2-normalized sentence embeddings, computed in batches."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = embed_tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
        with torch.no_grad():
            output = embed_model(**encoded)
        tokens = output.last_hidden_state
        mask = encoded['attention_mask'].unsqueeze(-1).expand(tokens.size()).float()
        pooled = torch.sum(tokens * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)

document_embeddings = get_text_embeddings(documents_df['text'].tolist())
train_df_cleaned['reference_answer_embedding'] = list(get_text_embeddings(train_df_cleaned['reference_answer'].tolist()))

print("Embeddings ready:", document_embeddings.shape)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings ready: (24, 384)


In [10]:
def retrieve_document(question, topic=None, crop=None, agro_zone=None, top_k=1):
    """Topic-filtered retrieval (mirrors the challenge's own baseline strategy): first
    narrow candidates by matching metadata, then break ties by embedding similarity to
    the question. Falls back to embedding similarity over ALL documents if nothing matches."""
    candidates = documents_df
    for col, val in (('topic', topic), ('crop', crop), ('agro_zone', agro_zone)):
        if val is not None and (candidates[col] == val).any():
            candidates = candidates[candidates[col] == val]

    query_emb = get_text_embeddings([question])[0]
    cand_idx = candidates.index.to_numpy()
    sims = document_embeddings[cand_idx] @ query_emb
    best_local = int(np.argmax(sims))
    best_doc_id = candidates.iloc[best_local]['document_id']
    return doc_by_id.loc[best_doc_id, 'text'], best_doc_id


## 5. Load the GPT-2 transformer

In [11]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')

if gpt2_tokenizer.pad_token is None:
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
    gpt2_model.config.pad_token_id = gpt2_tokenizer.pad_token_id

print("GPT-2 tokenizer and model loaded.")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 tokenizer and model loaded.


## 6. Train the model

Each training example is grounded in the factsheet it was written from (via
`document_id`), so the model learns to answer *from context* rather than memorizing
answers with no supporting text -- this matches how it will have to work at test time
(step 8), where the context comes from retrieval instead of a known `document_id`.


In [12]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

train_split_df, val_split_df = train_test_split(train_df_cleaned, test_size=0.15, random_state=SEED)
train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)
print(f"Train split: {len(train_split_df)} rows | Validation split: {len(val_split_df)} rows")

def build_prompt(context, question, answer=None):
    prompt = f"Context: {context}\n\nQuestion: {question}\nAnswer:"
    if answer is not None:
        prompt += f" {answer}{gpt2_tokenizer.eos_token}"
    return prompt

class QADataset(Dataset):
    def __init__(self, df, max_length):
        self.max_length = max_length
        self.texts = []
        for _, row in df.iterrows():
            context = doc_by_id.loc[row['document_id'], 'text']
            self.texts.append(build_prompt(context, row['question'], answer=row['reference_answer']))

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = gpt2_tokenizer(
            self.texts[idx], truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt',
        )
        input_ids = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_dataset = QADataset(train_split_df, MAX_LENGTH)
val_dataset = QADataset(val_split_df, MAX_LENGTH)

print("Example formatted training prompt:\n", train_dataset.texts[0])


Train split: 38 rows | Validation split: 7 rows
Example formatted training prompt:
 Context: Fall armyworm larvae feed in the whorl leaving ragged window-pane damage and frass pellets. Scout twice weekly at early vegetative stage and treat when more than ten percent of plants are infested. Rotate insecticide modes of action and conserve natural enemies along field margins.

Question: What damage does fall armyworm cause in maize?
Answer: Larvae feed in the whorl leaving ragged window-pane damage and frass.<|endoftext|>


In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='/content/slm_checkpoints',
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to=[],
    seed=SEED,
)

trainer = Trainer(
    model=gpt2_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

gpt2_model.save_pretrained('/content/slm_final')
gpt2_tokenizer.save_pretrained('/content/slm_final')
print("Fine-tuned SLM saved to /content/slm_final")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.398009,3.425128
2,3.483989,3.169155
3,3.051915,3.087329
4,2.724724,3.010917
5,2.518055,2.976144
6,2.414627,2.958801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned SLM saved to /content/slm_final


## 7. Evaluate the model

The challenge is scored by **mean Levenshtein distance** between predicted and reference
answers (lower is better), so that's the primary metric here -- not just training loss.
We also report mean embedding cosine similarity as a secondary, more forgiving signal.


In [14]:
def levenshtein(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev_row = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr_row = [i] + [0] * len(b)
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr_row[j] = min(prev_row[j] + 1, curr_row[j - 1] + 1, prev_row[j - 1] + cost)
        prev_row = curr_row
    return prev_row[-1]

def generate_answer(context, question, max_new_tokens=40):
    prompt = build_prompt(context, question)
    input_ids = gpt2_tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=MAX_LENGTH)
    output = gpt2_model.generate(
        input_ids, max_new_tokens=max_new_tokens, pad_token_id=gpt2_tokenizer.pad_token_id,
        do_sample=True, top_k=50, top_p=0.95, temperature=0.7,
    )
    decoded = gpt2_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.split("Answer:", 1)[-1].strip()

val_predictions = []
for _, row in val_split_df.iterrows():
    context = doc_by_id.loc[row['document_id'], 'text']
    val_predictions.append(generate_answer(context, row['question']))

lev_distances = [levenshtein(p, r) for p, r in zip(val_predictions, val_split_df['reference_answer'])]
val_pred_embeddings = get_text_embeddings(val_predictions)
val_ref_embeddings = np.array(val_split_df['reference_answer_embedding'].tolist())
similarity_scores = np.sum(val_pred_embeddings * val_ref_embeddings, axis=1)

print(f"Validation mean Levenshtein distance: {np.mean(lev_distances):.1f} (lower is better)")
print(f"Validation mean semantic similarity:  {similarity_scores.mean():.3f} (higher is better)")

print("\nSample predictions vs. references:")
for i in range(min(3, len(val_split_df))):
    print(f"\nQuestion:   {val_split_df.iloc[i]['question']}")
    print(f"Reference:  {val_split_df.iloc[i]['reference_answer']}")
    print(f"Generated:  {val_predictions[i]}")
    print(f"Levenshtein: {lev_distances[i]}  |  Similarity: {similarity_scores[i]:.3f}")


Validation mean Levenshtein distance: 61.7 (lower is better)
Validation mean semantic similarity:  0.463 (higher is better)

Sample predictions vs. references:

Question:   Sticky soot on seedlings in the nursery.
Reference:  Aphid honeydew leading to sooty mould; wash aphids and protect natural enemies.
Generated:  Dry out and use reflective mulch.
Levenshtein: 64  |  Similarity: 0.285

Question:   Poor bean pods despite flowers — micronutrient?
Reference:  Check boron deficiency; foliar boron only at recommended rates.
Generated:  Yellow phosphorous deficiency on sandy acidic soils and poor pod set on sandy acidic soils.
Levenshtein: 59  |  Similarity: 0.195

Question:   Drip emitters clogging — prevention?
Reference:  Filter water and flush lines weekly especially in sandy soils.
Generated:  Mulch beds to keep soil moisture stable between irrigation cycles.
Question: Cut evaporation — mulch to keep moisture stable between irrigation cycles.
Levenshtein: 119  |  Similarity: 0.404


## 8. Make predictions

Test questions don't have a `document_id`, so each one is grounded using the
topic-filtered document retrieval from step 4 before generation.


In [15]:
test_predictions = []
retrieved_doc_ids = []
for _, row in test_df_cleaned.iterrows():
    context, doc_id = retrieve_document(row['question'], row.get('topic'), row.get('crop'), row.get('agro_zone'))
    retrieved_doc_ids.append(doc_id)
    test_predictions.append(generate_answer(context, row['question']))

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL].values,
    ANSWER_COL: test_predictions,
})

submission_path = '/content/submission.csv'
submission.to_csv(submission_path, index=False)
print(f"Saved predictions for {len(submission)} test questions to '{submission_path}'.")
display(submission)


Saved predictions for 12 test questions to '/content/submission.csv'.


,QuestionId,Answer
0,1001,Split nitrogen between planting and knee-high ...
1,1002,Use dry feeders or feed racks spread out on ra...
2,1003,Inspect plots and use certified clean seed.
3,1004,Combine manure with a modest top-dress.\nQuest...
4,1005,Apply five to ten tonnes per hectare before pl...
5,1006,"Plant early with the rains, plant early with t..."
6,1007,Apply lime based on soil test before planting ...
7,1008,Stem borers tunnel inside sorghum stalks causi...
8,1009,Fix.
9,1010,Stabilize planting with rainfall forecasts or ...


In [16]:
submission_df = pd.read_csv('/content/submission.csv')
display(submission_df.head())

,QuestionId,Answer
0,1001,Split nitrogen between planting and knee-high ...
1,1002,Use dry feeders or feed racks spread out on ra...
2,1003,Inspect plots and use certified clean seed.
3,1004,Combine manure with a modest top-dress.\nQuest...
4,1005,Apply five to ten tonnes per hectare before pl...
